# 3 & 4번 분석: 평점 & 가격과 베스트셀러 지속 기간 분석

**분석 파트:**
- **3번**: 평점(Amazon Rating)과 베스트셀러 지속 기간 분석
- **4번**: 가격(Price)과 베스트셀러 지속 기간 분석

> ⚠️ **분석의 한계**: 본 데이터셋에는 실제 판매 수량이나 판매량 변수가 포함되어 있지 않으므로,  
> 본 분석의 목적은 판매 예측이 아니라 **평점·가격과 베스트셀러 지속 기간의 관계를 확인**하는 데 있습니다.

## 0. 환경 설정

분석에 필요한 라이브러리를 불러오고, 한글 폰트 및 경로·출력 폴더를 설정합니다.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# 한글 폰트 설정
def set_korean_font():
    from matplotlib import font_manager
    installed = {f.name for f in font_manager.fontManager.ttflist}
    candidates = ["AppleGothic", "Malgun Gothic", "NanumGothic",
                  "Noto Sans CJK KR", "Noto Sans KR"]
    for name in candidates:
        if name in installed:
            plt.rcParams["font.family"] = name
            break
    plt.rcParams["axes.unicode_minus"] = False

set_korean_font()
sns.set_style("whitegrid")
matplotlib.rcParams["figure.dpi"] = 110

# 경로 설정
BASE_DIR = Path(".").resolve().parent
DATA_DIR = BASE_DIR / "data" / "processed"

amazon_clean_path = DATA_DIR / "amazon_clean.csv"
final_books_path  = DATA_DIR / "final_books_dataset.csv"

# 출력 폴더 생성
FIG_DIR = BASE_DIR / "outputs" / "rating_price_visualization"
RES_DIR = BASE_DIR / "outputs" / "rating_price_analysis"
FIG_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)

def savefig(fig, name):
    path = FIG_DIR / name
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    print(f"  - 그래프 저장: {path.name}")

# 상관계수 결과를 모아 마지막에 한 번에 저장하기 위한 리스트
correlation_records = []

def add_corr(label, x, y, n):
    r = float(np.corrcoef(x, y)[0, 1])
    correlation_records.append({"분석": label, "표본수": int(n),
                                "피어슨_상관계수": round(r, 4)})
    return r

print("환경 설정 완료")

## 1. 데이터 불러오기 및 기본 구조 확인

분석에 사용할 두 가지 데이터를 불러옵니다.

| 파일 | 설명 |
|------|------|
| `amazon_clean.csv` | 연도별 베스트셀러 기록 데이터 (연도·도서 단위) |
| `final_books_dataset.csv` | 도서 단위 요약 데이터 (도서별 통계치 집약) |

In [ ]:
amazon_clean = pd.read_csv(amazon_clean_path, encoding="utf-8-sig")
final_books  = pd.read_csv(final_books_path,  encoding="utf-8-sig")

print("[amazon_clean.csv]  (연도별 베스트셀러 기록 데이터)")
print(f"  크기: {amazon_clean.shape[0]}행 x {amazon_clean.shape[1]}열")
print(f"  결측치 합계: {int(amazon_clean.isna().sum().sum())}개")
print(f"  중복 행: {int(amazon_clean.duplicated().sum())}개")
print(f"  가격(price) 0인 기록: {int((amazon_clean['price'] == 0).sum())}건")

print()
print("[final_books_dataset.csv]  (도서 단위 요약 데이터)")
print(f"  크기: {final_books.shape[0]}행 x {final_books.shape[1]}열")
print(f"  결측치 합계: {int(final_books.isna().sum().sum())}개")
print(f"  중복 행: {int(final_books.duplicated().sum())}개")
print(f"  평균 가격(avg_price) 0인 도서: {int((final_books['avg_price'] == 0).sum())}건")
print(f"  단년(0) / 다년(1) 베스트셀러: {final_books['is_multi_year_bestseller'].value_counts().to_dict()}")
print(f"  비장기(0) / 장기(1) 베스트셀러: {final_books['is_long_seller'].value_counts().to_dict()}")

In [ ]:
amazon_clean.head()

In [ ]:
final_books.head()

## 2. 분석 변수 생성

연속형 변수인 **평점**과 **가격**을 구간(범주형)으로 변환하여 그룹별 비율 분석에 활용합니다.

| 구간 변수 | 기준 |
|-----------|------|
| 평점 구간 | 4.0 미만 / 4.0~4.3 / 4.4~4.6 / 4.7 이상 |
| 가격 구간 | 5달러 이하 / 6~10달러 / 11~15달러 / 16달러 이상 |

In [ ]:
RATING_LABELS = ["4.0 미만", "4.0~4.3", "4.4~4.6", "4.7 이상"]
def make_rating_group(s):
    return pd.cut(s, bins=[-np.inf, 4.0, 4.4, 4.7, np.inf],
                  right=False, labels=RATING_LABELS)

PRICE_LABELS = ["5달러 이하", "6~10달러", "11~15달러", "16달러 이상"]
def make_price_group(s):
    return pd.cut(s, bins=[-np.inf, 5, 10, 15, np.inf],
                  right=True, labels=PRICE_LABELS)

print("평점 구간 라벨:", RATING_LABELS)
print("가격 구간 라벨:", PRICE_LABELS)

---
## 3. 평점과 베스트셀러 지속 기간 분석

**분석 질문**: 평점이 높은 도서가 반드시 장기 베스트셀러가 되는가?

**분석 내용:**
- 평점 전체 분포 확인
- 장르별 평균 평점 비교
- 단년 vs 다년 베스트셀러 평균 평점 비교
- 평점과 베스트셀러 등장 횟수 / 지속 기간의 상관관계
- 평점 구간별 다년·장기 베스트셀러 비율

### 3.1 평점 분포 확인

전체 베스트셀러 데이터에서 Amazon 사용자 평점이 어떻게 분포하는지 히스토그램으로 확인합니다.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(amazon_clean["amazon_rating"], bins=20, color="#4C72B0", edgecolor="white")
ax.set_title("평점 분포", fontsize=14)
ax.set_xlabel("Amazon 사용자 평점")
ax.set_ylabel("도서 기록 수")
savefig(fig, "rating_distribution.png")

print(f"  평점 평균  : {amazon_clean['amazon_rating'].mean():.3f}")
print(f"  평점 중앙값: {amazon_clean['amazon_rating'].median():.3f}")
print(f"  최소 ~ 최대: {amazon_clean['amazon_rating'].min()} ~ {amazon_clean['amazon_rating'].max()}")

### 3.2 장르별 평균 평점 비교

Fiction과 Non Fiction 간 평균 평점 차이를 비교합니다.

In [ ]:
genre_rating = amazon_clean.groupby("genre")["amazon_rating"].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(genre_rating.index.astype(str), genre_rating.values, color=["#4C72B0", "#DD8452"])
ax.set_title("장르별 평균 평점 비교", fontsize=14)
ax.set_xlabel("장르")
ax.set_ylabel("평균 평점")
ax.set_ylim(genre_rating.min() - 0.1, genre_rating.max() + 0.1)
for i, v in enumerate(genre_rating.values):
    ax.text(i, v + 0.005, f"{v:.3f}", ha="center", va="bottom")
savefig(fig, "avg_rating_by_genre.png")

print("장르별 평균 평점:", {k: round(v, 3) for k, v in genre_rating.items()})

### 3.3 단년 vs 다년 베스트셀러 평균 평점 비교

한 해만 베스트셀러에 오른 도서(단년)와 여러 해 오른 도서(다년)의 평균 평점 차이를 비교합니다.

In [ ]:
label_map_my = {0: "단년 베스트셀러", 1: "다년 베스트셀러"}
fb_r = final_books.copy()
fb_r["status"] = fb_r["is_multi_year_bestseller"].map(label_map_my)

rating_my_summary = (fb_r.groupby("status")["avg_rating"]
                     .agg(도서수="count", 평균="mean", 중앙값="median", 표준편차="std")
                     .reindex(["단년 베스트셀러", "다년 베스트셀러"])
                     .round(4))
print(rating_my_summary)
rating_my_summary.to_csv(RES_DIR / "rating_multi_year_summary.csv", encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(7, 5))
order = ["단년 베스트셀러", "다년 베스트셀러"]
vals  = [rating_my_summary.loc[o, "평균"] for o in order]
ax.bar(order, vals, color=["#999999", "#4C72B0"])
ax.set_title("단년·다년 베스트셀러 평균 평점 비교", fontsize=14)
ax.set_xlabel("베스트셀러 유형")
ax.set_ylabel("평균 평점(avg_rating)")
ax.set_ylim(min(vals) - 0.1, max(vals) + 0.1)
for i, v in enumerate(vals):
    ax.text(i, v + 0.005, f"{v:.3f}", ha="center", va="bottom")
savefig(fig, "avg_rating_by_multi_year_status.png")

### 3.4 평점과 베스트셀러 등장 횟수의 관계

도서의 평균 평점과 베스트셀러 목록에 등장한 횟수 사이에 선형 상관관계가 있는지 산점도와 피어슨 상관계수로 확인합니다.

In [ ]:
r_appear = add_corr("평점(avg_rating) vs 등장횟수(appear_count)",
                    final_books["avg_rating"], final_books["appear_count"],
                    len(final_books))

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(final_books["avg_rating"], final_books["appear_count"],
           alpha=0.4, color="#4C72B0", edgecolor="none")
ax.set_title("평점과 베스트셀러 등장 횟수의 관계", fontsize=14)
ax.set_xlabel("평균 평점(avg_rating)")
ax.set_ylabel("베스트셀러 등장 횟수(appear_count)")
ax.text(0.02, 0.95, f"피어슨 상관계수 = {r_appear:.3f}",
        transform=ax.transAxes, va="top",
        bbox=dict(boxstyle="round", fc="white", ec="#cccccc"))
savefig(fig, "rating_vs_appear_count.png")
print(f"  피어슨 상관계수: {r_appear:.4f}")

### 3.5 평점과 베스트셀러 지속 기간의 관계

도서의 평균 평점과 베스트셀러 지속 기간(첫 등장 ~ 마지막 등장 연도 차이) 사이의 상관관계를 확인합니다.

In [ ]:
r_dur = add_corr("평점(avg_rating) vs 지속기간(duration)",
                 final_books["avg_rating"], final_books["duration"],
                 len(final_books))

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(final_books["avg_rating"], final_books["duration"],
           alpha=0.4, color="#DD8452", edgecolor="none")
ax.set_title("평점과 베스트셀러 지속 기간의 관계", fontsize=14)
ax.set_xlabel("평균 평점(avg_rating)")
ax.set_ylabel("베스트셀러 지속 기간(duration)")
ax.text(0.02, 0.95, f"피어슨 상관계수 = {r_dur:.3f}",
        transform=ax.transAxes, va="top",
        bbox=dict(boxstyle="round", fc="white", ec="#cccccc"))
savefig(fig, "rating_vs_duration.png")
print(f"  피어슨 상관계수: {r_dur:.4f}")

### 3.6 & 3.7 평점 구간별 다년·장기 베스트셀러 비율

평점을 4개 구간으로 나눈 뒤, 각 구간에서 **다년 베스트셀러** 및 **장기 베스트셀러**가 차지하는 비율을 계산합니다.

In [ ]:
fb_rg = final_books.copy()
fb_rg["rating_group"] = make_rating_group(fb_rg["avg_rating"])

rating_group_summary = (fb_rg.groupby("rating_group", observed=False)
                        .agg(도서수=("avg_rating", "count"),
                             다년비율=("is_multi_year_bestseller", "mean"),
                             장기비율=("is_long_seller", "mean"))
                        .reindex(RATING_LABELS))
rating_group_summary["다년비율"] = (rating_group_summary["다년비율"] * 100).round(2)
rating_group_summary["장기비율"] = (rating_group_summary["장기비율"] * 100).round(2)
print(rating_group_summary)
rating_group_summary.to_csv(RES_DIR / "rating_group_summary.csv", encoding="utf-8-sig")

In [ ]:
# 3.6 다년 베스트셀러 비율
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(RATING_LABELS, rating_group_summary["다년비율"].values, color="#4C72B0")
ax.set_title("평점 구간별 다년 베스트셀러 비율", fontsize=14)
ax.set_xlabel("평점 구간")
ax.set_ylabel("다년 베스트셀러 비율(%)")
for i, v in enumerate(rating_group_summary["다년비율"].values):
    if not np.isnan(v):
        ax.text(i, v + 0.5, f"{v:.1f}%", ha="center", va="bottom")
savefig(fig, "multi_year_ratio_by_rating_group.png")

# 3.7 장기 베스트셀러 비율
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(RATING_LABELS, rating_group_summary["장기비율"].values, color="#55A868")
ax.set_title("평점 구간별 장기 베스트셀러 비율", fontsize=14)
ax.set_xlabel("평점 구간")
ax.set_ylabel("장기 베스트셀러 비율(%)")
for i, v in enumerate(rating_group_summary["장기비율"].values):
    if not np.isnan(v):
        ax.text(i, v + 0.5, f"{v:.1f}%", ha="center", va="bottom")
savefig(fig, "long_seller_ratio_by_rating_group.png")

### 3.8 평점 분석 결론

**질문**: 평점이 높은 도서가 반드시 장기 베스트셀러가 되는가?

**결론**:  
평점이 높은 구간에서 다년 베스트셀러 비율이 높게 나타날 수 있지만,  
평점만으로 장기 베스트셀러 여부를 단정하기는 어렵다.  
특히 베스트셀러 도서 대부분이 이미 높은 평점을 가지고 있어 **평점의 구분력이 제한**될 수 있다.

---
## 4. 가격과 베스트셀러 지속 기간 분석

**분석 질문**: 가격이 낮은 도서가 장기 베스트셀러가 될 가능성이 높은가?

**분석 내용:**
- 가격 0인 데이터 처리
- 가격 전체 분포 확인
- 장르별 평균 가격 비교
- 가격과 리뷰 수 / 등장 횟수 / 지속 기간의 상관관계
- 단년 vs 다년 베스트셀러 평균 가격 비교
- 가격 구간별 다년·장기 베스트셀러 비율

### 4.1 가격 0인 데이터 처리

가격이 0인 기록은 무료 도서이거나 데이터 오류일 가능성이 있으므로 분석에서 제외합니다.

In [ ]:
n_zero_clean = int((amazon_clean["price"] == 0).sum())
n_zero_final = int((final_books["avg_price"] == 0).sum())

amazon_price = amazon_clean[amazon_clean["price"] > 0].copy()
final_price  = final_books[final_books["avg_price"] > 0].copy()

print(f"amazon_clean : price==0 {n_zero_clean}건 제외 → {len(amazon_price)}건 사용")
print(f"final_books  : avg_price==0 {n_zero_final}건 제외 → {len(final_price)}권 사용")

### 4.2 가격 분포 확인

0을 제외한 베스트셀러 도서들의 가격 분포를 히스토그램으로 확인합니다.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(amazon_price["price"], bins=30, color="#DD8452", edgecolor="white")
ax.set_title("가격 분포", fontsize=14)
ax.set_xlabel("가격(달러)")
ax.set_ylabel("도서 기록 수")
savefig(fig, "price_distribution.png")

print(f"  가격 평균  : {amazon_price['price'].mean():.2f}")
print(f"  가격 중앙값: {amazon_price['price'].median():.2f}")
print(f"  최소 ~ 최대: {amazon_price['price'].min()} ~ {amazon_price['price'].max()}")

### 4.3 장르별 평균 가격 비교

Fiction과 Non Fiction 장르 간 평균 가격 차이를 비교합니다.

In [ ]:
genre_price = amazon_price.groupby("genre")["price"].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(genre_price.index.astype(str), genre_price.values, color=["#4C72B0", "#DD8452"])
ax.set_title("장르별 평균 가격 비교", fontsize=14)
ax.set_xlabel("장르")
ax.set_ylabel("평균 가격(달러)")
for i, v in enumerate(genre_price.values):
    ax.text(i, v + 0.1, f"{v:.2f}", ha="center", va="bottom")
savefig(fig, "avg_price_by_genre.png")

print("장르별 평균 가격:", {k: round(v, 2) for k, v in genre_price.items()})

### 4.4 가격과 리뷰 수의 관계

리뷰 수는 우편향이 강하므로 **log1p 변환**한 값으로도 상관계수를 계산하여 비교합니다.

In [ ]:
review_log = np.log1p(amazon_price["amazon_reviews"])

r_price_rev    = add_corr("가격(price) vs 리뷰수(amazon_reviews)",
                          amazon_price["price"], amazon_price["amazon_reviews"], len(amazon_price))
r_price_revlog = add_corr("가격(price) vs 리뷰수 로그(review_log)",
                          amazon_price["price"], review_log, len(amazon_price))

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(amazon_price["price"], review_log, alpha=0.4, color="#4C72B0", edgecolor="none")
ax.set_title("가격과 리뷰 수의 관계", fontsize=14)
ax.set_xlabel("가격(달러)")
ax.set_ylabel("리뷰 수(log1p 변환)")
ax.text(0.02, 0.95,
        f"상관계수(원자료) = {r_price_rev:.3f}\n상관계수(로그) = {r_price_revlog:.3f}",
        transform=ax.transAxes, va="top",
        bbox=dict(boxstyle="round", fc="white", ec="#cccccc"))
savefig(fig, "price_vs_reviews.png")

print(f"  가격 vs 리뷰수(원자료) 상관계수: {r_price_rev:.4f}")
print(f"  가격 vs 리뷰수(로그변환) 상관계수: {r_price_revlog:.4f}")

### 4.5 가격과 베스트셀러 등장 횟수의 관계

도서의 평균 가격과 베스트셀러 목록 등장 횟수 사이의 상관관계를 확인합니다.

In [ ]:
r_price_appear = add_corr("평균가격(avg_price) vs 등장횟수(appear_count)",
                          final_price["avg_price"], final_price["appear_count"], len(final_price))

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(final_price["avg_price"], final_price["appear_count"],
           alpha=0.4, color="#DD8452", edgecolor="none")
ax.set_title("가격과 베스트셀러 등장 횟수의 관계", fontsize=14)
ax.set_xlabel("평균 가격(avg_price, 달러)")
ax.set_ylabel("베스트셀러 등장 횟수(appear_count)")
ax.text(0.02, 0.95, f"피어슨 상관계수 = {r_price_appear:.3f}",
        transform=ax.transAxes, va="top",
        bbox=dict(boxstyle="round", fc="white", ec="#cccccc"))
savefig(fig, "price_vs_appear_count.png")
print(f"  피어슨 상관계수: {r_price_appear:.4f}")

### 4.6 가격과 베스트셀러 지속 기간의 관계

도서의 평균 가격과 베스트셀러 지속 기간 사이의 상관관계를 확인합니다.

In [ ]:
r_price_dur = add_corr("평균가격(avg_price) vs 지속기간(duration)",
                       final_price["avg_price"], final_price["duration"], len(final_price))

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(final_price["avg_price"], final_price["duration"],
           alpha=0.4, color="#4C72B0", edgecolor="none")
ax.set_title("가격과 베스트셀러 지속 기간의 관계", fontsize=14)
ax.set_xlabel("평균 가격(avg_price, 달러)")
ax.set_ylabel("베스트셀러 지속 기간(duration)")
ax.text(0.02, 0.95, f"피어슨 상관계수 = {r_price_dur:.3f}",
        transform=ax.transAxes, va="top",
        bbox=dict(boxstyle="round", fc="white", ec="#cccccc"))
savefig(fig, "price_vs_duration.png")
print(f"  피어슨 상관계수: {r_price_dur:.4f}")

### 4.7 단년 vs 다년 베스트셀러 평균 가격 비교

단년·다년 베스트셀러 간 평균 가격 차이를 요약 통계표와 막대그래프로 비교합니다.

In [ ]:
fp = final_price.copy()
fp["status"] = fp["is_multi_year_bestseller"].map(label_map_my)

price_my_summary = (fp.groupby("status")["avg_price"]
                    .agg(도서수="count", 평균="mean", 중앙값="median", 표준편차="std")
                    .reindex(["단년 베스트셀러", "다년 베스트셀러"])
                    .round(4))
print(price_my_summary)
price_my_summary.to_csv(RES_DIR / "price_multi_year_summary.csv", encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(7, 5))
order = ["단년 베스트셀러", "다년 베스트셀러"]
vals  = [price_my_summary.loc[o, "평균"] for o in order]
ax.bar(order, vals, color=["#999999", "#DD8452"])
ax.set_title("단년·다년 베스트셀러 평균 가격 비교", fontsize=14)
ax.set_xlabel("베스트셀러 유형")
ax.set_ylabel("평균 가격(avg_price, 달러)")
for i, v in enumerate(vals):
    ax.text(i, v + 0.1, f"{v:.2f}", ha="center", va="bottom")
savefig(fig, "avg_price_by_multi_year_status.png")

### 4.8 & 4.9 가격 구간별 다년·장기 베스트셀러 비율

가격을 4개 구간으로 나눈 뒤, 각 구간에서 **다년 베스트셀러** 및 **장기 베스트셀러**가 차지하는 비율을 계산합니다.

In [ ]:
fp_g = final_price.copy()
fp_g["price_group"] = make_price_group(fp_g["avg_price"])

price_group_summary = (fp_g.groupby("price_group", observed=False)
                       .agg(도서수=("avg_price", "count"),
                            다년비율=("is_multi_year_bestseller", "mean"),
                            장기비율=("is_long_seller", "mean"))
                       .reindex(PRICE_LABELS))
price_group_summary["다년비율"] = (price_group_summary["다년비율"] * 100).round(2)
price_group_summary["장기비율"] = (price_group_summary["장기비율"] * 100).round(2)
print(price_group_summary)
price_group_summary.to_csv(RES_DIR / "price_group_summary.csv", encoding="utf-8-sig")

In [ ]:
# 4.8 다년 베스트셀러 비율
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(PRICE_LABELS, price_group_summary["다년비율"].values, color="#DD8452")
ax.set_title("가격 구간별 다년 베스트셀러 비율", fontsize=14)
ax.set_xlabel("가격 구간")
ax.set_ylabel("다년 베스트셀러 비율(%)")
for i, v in enumerate(price_group_summary["다년비율"].values):
    if not np.isnan(v):
        ax.text(i, v + 0.5, f"{v:.1f}%", ha="center", va="bottom")
savefig(fig, "multi_year_ratio_by_price_group.png")

# 4.9 장기 베스트셀러 비율
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(PRICE_LABELS, price_group_summary["장기비율"].values, color="#C44E52")
ax.set_title("가격 구간별 장기 베스트셀러 비율", fontsize=14)
ax.set_xlabel("가격 구간")
ax.set_ylabel("장기 베스트셀러 비율(%)")
for i, v in enumerate(price_group_summary["장기비율"].values):
    if not np.isnan(v):
        ax.text(i, v + 0.5, f"{v:.1f}%", ha="center", va="bottom")
savefig(fig, "long_seller_ratio_by_price_group.png")

### 4.10 가격 분석 결론

**질문**: 가격이 낮은 도서가 장기 베스트셀러가 될 가능성이 높은가?

**결론**:  
일부 가격 구간에서 다년 베스트셀러 비율이 높게 나타날 수 있지만,  
가격이 낮다고 해서 반드시 장기 베스트셀러가 된다고 보기는 어렵다.  
가격은 **장르, 독자층, 리뷰 수, 출판 시기** 등 다른 요인과 함께 해석해야 한다.

---
## 5. 상관계수 요약 및 분석 완료

분석 과정에서 계산된 모든 피어슨 상관계수를 하나의 표로 정리하고 CSV로 저장합니다.

In [ ]:
corr_df = pd.DataFrame(correlation_records)
corr_df.to_csv(RES_DIR / "correlation_summary.csv", index=False, encoding="utf-8-sig")

print("=" * 70)
print("상관계수 요약 (correlation_summary.csv)")
print("=" * 70)
print(corr_df.to_string(index=False))

print()
print(f"  그래프 저장 위치 : {FIG_DIR}")
print(f"  결과표 저장 위치 : {RES_DIR}")

print()
print("[분석 한계] 본 데이터셋에는 실제 판매 수량이나 판매량 변수가 포함되어 있지 않으므로,")
print("           본 분석은 판매 예측이 아니라 평점·가격과 베스트셀러 지속 기간의 관계를")
print("           확인하는 데 목적을 둡니다.")

In [ ]:
corr_df